# Kaggle GPU run — generation, claims and judge (M5–M7)

See `KAGGLE.md` for the one-time setup (dataset upload, GPU, Internet, `HF_TOKEN` secret).

Run with **Save Version → Save & Run All (Commit)** so it keeps running in the background (up to 12 h). Everything is cached: if a run stops, add this notebook's previous output as an input and run again — finished records are skipped.

When it finishes, download `ghd_outputs.zip` from the *Output* tab and merge it locally:
`python -m groundedgen.kaggle merge ghd_outputs.zip configs/local.yaml`

In [ ]:
# Settings for this run.
SPLITS = ["dev", "test"]   # "test" only after M8 (frozen.json); the code refuses otherwise.
                           # dev is cached, so it only fills in whatever is missing.
CONDITIONS = ["A", "A-F", "B", "B-Abs", "APG", "C", "C-Mask"]
RUN_JUDGE = True
REPO_URL = "https://github.com/Yashas-BR/grounded-product-description.git"

In [ ]:
# Code and dependencies (needs Internet on in the notebook settings).
import os, subprocess, sys
CODE_DIR = "/kaggle/working/ghd"
if os.path.isdir(CODE_DIR):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "bitsandbytes>=0.43", "sentencepiece", "krippendorff", "accelerate"], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])

In [ ]:
# Restore the project from /kaggle/input (uploaded dataset + any earlier run outputs).
import json, logging
from groundedgen import kaggle, utils

logging.basicConfig(level=logging.INFO)
cfg = utils.load_config(extra_paths=["configs/kaggle.yaml"])
utils.apply_runtime_env(cfg)
print(kaggle.restore(cfg["project_dir"]))
assert (utils.project_path(cfg, "data", "manifest.csv")).exists(), "Add the ghd-project dataset as an input"
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)

In [ ]:
utils.set_seed(cfg["seed"])
assert utils.load_hf_token(), "Add HF_TOKEN under Add-ons → Secrets (needed for Mistral)"
env = utils.capture_env()
print(json.dumps(env, indent=2))
utils.update_run_info(utils.project_path(cfg, "outputs", "run_info.json"), {"env": env})

## Generation

In [ ]:
from groundedgen import generate

for SPLIT in SPLITS:
    summary = generate.run_matrix(cfg, SPLIT, CONDITIONS)
    print(SPLIT, json.dumps(summary, indent=2))
    assert all(v is None or v < 15 for v in summary["peak_gpu_gb"].values()), summary["peak_gpu_gb"]

## Claims and judge

In [ ]:
from groundedgen import claims, judge

for gen_file in sorted(utils.project_path(cfg, "outputs", "generation").rglob("*.jsonl")):
    claims.run_claims(cfg["project_dir"], gen_file)
if RUN_JUDGE:
    for SPLIT in SPLITS:
        print(SPLIT, json.dumps(judge.run_all(cfg, SPLIT), indent=2))

In [ ]:
# APG informativeness cost and judge parse health per split.
import pandas as pd
from groundedgen import data
for SPLIT in SPLITS:
    print("==", SPLIT)
    ids = set(data.load_manifest(cfg, splits=[SPLIT])["item_id"])
    rows = []
    for path in utils.project_path(cfg, "outputs", "generation", "APG").glob("*.jsonl"):
        for r in utils.read_jsonl(path):
            if r["item_id"] in ids:
                log = r["apg"]
                rows.append({"file": path.stem, "fallback": log["fallback_used"],
                             "dropped_share": (log["sentences_dropped_prune"] + log["sentences_dropped_verify"])
                                              / max(1, log["sentences_total"])})
    if rows:
        display(pd.DataFrame(rows).groupby("file").mean())
    if RUN_JUDGE:
        print(judge.ucr(judge.judged_descriptions(cfg, SPLIT).to_dict("records")))

## Package results for download

In [ ]:
out = kaggle.pack(cfg["project_dir"], "/kaggle/working/ghd_outputs.zip", dirs=("outputs",))
print(out, round(out.stat().st_size / 2**20, 1), "MB")